# Pratica 12 — Multicollinearità, selezione delle variabili e validazione

**Corso:** Statistica Applicata con Python · **Blocco 4:** Modelli ed esperimenti · **Tempo stimato:** 60 minuti
**Lezione associata:** `notebooks_clase/clase12_multicolinealidad_seleccion_validacion_IT.ipynb`

## Come funziona questa pratica

- Ogni esercizio ha una cella con `# TODO`: completala.
- Alla fine di ogni esercizio c'è un **controllo automatico** (`assert`). Se fallisce, il messaggio ti indica cosa rivedere. Se dimentichi di completare un `TODO`, il messaggio te lo ricorda.
- Gli esercizi con simulazione o con dati casuali usano tolleranze basate sull'errore standard, quindi un risultato corretto non dovrebbe fallire per caso.
- Le **soluzioni** sono alla fine del notebook, in una sezione a parte. **Non aprirle prima di aver provato.**
- Esegui le celle in ordine. Tutti i dati sono sintetici: *esempi illustrativi*.

---
## Cella 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings("ignore")

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

# Coefficienti VERI con cui generiamo i dati (li conosciamo solo perché sono sintetici)
VERDAD = {"Intercept": -1.8, "antiguedad": -0.03, "gasto": 0.03, "datos_gb": 0.0, "tickets": 0.45, "plan_anual": -0.9}

def generar_clientes(n, rng):
    """Genera n clienti sintetici (esempio illustrativo) con abbandono secondo VERDAD."""
    antiguedad = np.clip(rng.gamma(2.0, 14, n), 1, 72).round()
    gasto = np.clip(30 + 0.25 * antiguedad + rng.normal(0, 10, n), 10, None).round(1)
    datos_gb = np.clip(0.09 * gasto + rng.normal(0, 0.25, n), 0.1, None).round(2)   # molto correlata con gasto
    tickets = rng.poisson(np.exp(0.008 * (gasto - 40)))
    plan_anual = rng.binomial(1, 0.35, n)
    eta = (VERDAD["Intercept"] + VERDAD["antiguedad"] * antiguedad + VERDAD["gasto"] * gasto
           + VERDAD["tickets"] * tickets + VERDAD["plan_anual"] * plan_anual)
    abandona = rng.binomial(1, 1 / (1 + np.exp(-eta)))
    return pd.DataFrame({"abandona": abandona, "antiguedad": antiguedad, "gasto": gasto,
                         "datos_gb": datos_gb, "tickets": tickets, "plan_anual": plan_anual})

def log_loss(y, p):
    """Perdita logaritmica media (minore è meglio)."""
    y, p = np.asarray(y), np.asarray(p)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def exactitud(y, p, umbral=0.5):
    """Proporzione di risposte corrette classificando con una soglia."""
    return float(np.mean((np.asarray(p) > umbral) == np.asarray(y)))

print("Librerie e funzioni caricate correttamente.")

## Esercizio 1 — Multicollinearità: VIF e quale variabile eliminare

Con `datos` (1 500 clienti) calcola il VIF delle cinque variabili esplicative (con costante, senza includere la costante nel risultato): `vifs` è un dizionario `{variabile: VIF}`. Decidi quale variabile eliminare dalla coppia correlata (`var_a_quitar`: per senso di business, elimina quella che non è la fatturazione) e riadatta: `se_con` è l'errore standard di `gasto` nel modello con le cinque variabili e `se_sin` nel modello senza la variabile eliminata.

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
columnas = ["antiguedad", "gasto", "datos_gb", "tickets", "plan_anual"]
X_e1 = sm.add_constant(datos[columnas])

vifs = None            # TODO: dizionario {variabile: VIF}, senza la costante
var_a_quitar = None    # TODO: nome della variabile (testo)
se_con = None          # TODO: errore standard di gasto con le cinque variabili
se_sin = None          # TODO: errore standard di gasto senza la variabile eliminata

# Controllo automatico
assert all(v is not None for v in (vifs, var_a_quitar, se_con, se_sin)), "Completa le righe contrassegnate con TODO"
assert set(vifs) == set(columnas), "vifs deve avere una voce per variabile, senza la costante"
ref = {c: variance_inflation_factor(X_e1.values, i) for i, c in enumerate(X_e1.columns) if c != "const"}
assert all(np.isclose(vifs[c], ref[c]) for c in columnas), "usa variance_inflation_factor(X.values, i) sulla matrice con costante"
assert var_a_quitar == "datos_gb", "elimina datos_gb: gasto è la fatturazione, che l'azienda controlla"
ref_con = smf.logit("abandona ~ antiguedad + gasto + datos_gb + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
ref_sin = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
assert np.isclose(se_con, ref_con) and np.isclose(se_sin, ref_sin), "adatta i due modelli con smf.logit e prendi .bse['gasto']"
assert se_sin < se_con / 2, "eliminando la variabile ridondante l'errore standard dovrebbe scendere molto"
print({k: round(v, 1) for k, v in vifs.items()})
print(f"Errore standard di gasto: {se_con:.4f} → {se_sin:.4f}")
print("Esercizio 1 corretto ✔")

## Esercizio 2 — AIC e BIC a mano tra modelli candidati

Con 500 clienti (`datos2`) e tre modelli candidati già adattati (`ajustes2`), calcola **a mano** con la log-verosimiglianza `m.llf`, il numero di parametri `k = len(m.params)` e `n = m.nobs`: `aic_mano` (`2k − 2·LL`) e `bic_mano` (`k·ln(n) − 2·LL`), due dizionari `{nome: valore}`. Poi `elegido_aic` e `elegido_bic` sono il nome del modello con valore minore in ciascun criterio, e `dif_irrelevante` è `True` se M2 e M3 differiscono di meno di 2 punti di AIC.

In [ ]:
datos2 = generar_clientes(500, np.random.default_rng(100))
formulas2 = {"M2": "abandona ~ antiguedad + tickets + plan_anual",
             "M3": "abandona ~ antiguedad + tickets + plan_anual + gasto",
             "M4": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}
ajustes2 = {nombre: smf.logit(f, data=datos2).fit(disp=0) for nombre, f in formulas2.items()}

aic_mano = None         # TODO: dict {nome: 2k - 2LL}
bic_mano = None         # TODO: dict {nome: k·ln(n) - 2LL}
elegido_aic = None      # TODO
elegido_bic = None      # TODO
dif_irrelevante = None  # TODO

# Controllo automatico
assert all(v is not None for v in (aic_mano, bic_mano, elegido_aic, elegido_bic, dif_irrelevante)), "Completa le righe contrassegnate con TODO"
assert all(np.isclose(aic_mano[k], ajustes2[k].aic) for k in formulas2), "AIC = 2k - 2LL, con k = len(m.params) (include la costante)"
assert all(np.isclose(bic_mano[k], ajustes2[k].bic) for k in formulas2), "BIC = k·ln(n) - 2LL, con n = m.nobs"
assert elegido_aic == min(ajustes2, key=lambda k: ajustes2[k].aic) and elegido_bic == min(ajustes2, key=lambda k: ajustes2[k].bic), "scegli il nome con il valore minore di ciascun criterio"
assert elegido_aic == "M3" and elegido_bic == "M2", "in questo campione l'AIC preferisce M3 e il BIC preferisce il più semplice, M2"
assert dif_irrelevante == True, "gli AIC di M2 e M3 differiscono di meno di 2 punti"
print("AIC:", {k: round(v, 1) for k, v in aic_mano.items()}, "→", elegido_aic)
print("BIC:", {k: round(v, 1) for k, v in bic_mano.items()}, "→", elegido_bic)
print("Esercizio 2 corretto ✔")

## Esercizio 3 — Hold-out a mano: addestramento, test e overfitting

Con 300 clienti e **15 variabili di rumore** aggiunte, separa il 70 % per l'addestramento (`train`) e il 30 % per il test (`test`) usando l'ordine casuale `orden` già calcolato. Adatta il modello con le 4 variabili reali più il rumore **solo con `train`** e calcola, con `log_loss` della Cella 0: `ll_train`, `ll_test` e `ll_base` (log-loss su `test` della previsione costante con il tasso di abbandono di `train`). Poi, `hay_sobreajuste` è `True` se il log-loss in test è peggiore (maggiore) che in addestramento.

In [ ]:
rng_e3 = np.random.default_rng(77)
datos3 = generar_clientes(300, rng_e3)
for j in range(15):
    datos3[f"nz{j + 1}"] = rng_e3.normal(size=300)
formula3 = "abandona ~ antiguedad + gasto + tickets + plan_anual" + "".join(f" + nz{j + 1}" for j in range(15))
orden = rng_e3.permutation(300)

train = None            # TODO: prime 210 righe secondo orden (datos3.iloc[orden[:210]])
test = None             # TODO: le 90 restanti
ll_train = None         # TODO
ll_test = None          # TODO
ll_base = None          # TODO
hay_sobreajuste = None  # TODO

# Controllo automatico
assert all(v is not None for v in (train, test, ll_train, ll_test, ll_base, hay_sobreajuste)), "Completa le righe contrassegnate con TODO"
assert len(train) == 210 and len(test) == 90, "70 % per l'addestramento (210) e 30 % per il test (90)"
assert set(train.index).isdisjoint(test.index), "addestramento e test non devono condividere righe"
m3_ref = smf.logit(formula3, data=datos3.iloc[orden[:210]]).fit(disp=0, maxiter=200)
assert np.isclose(ll_train, log_loss(datos3.iloc[orden[:210]]["abandona"], m3_ref.predict(datos3.iloc[orden[:210]]))), "ll_train: log_loss(train['abandona'], modelo.predict(train))"
assert np.isclose(ll_test, log_loss(datos3.iloc[orden[210:]]["abandona"], m3_ref.predict(datos3.iloc[orden[210:]]))), "il modello si adatta solo con train e si valuta su test"
tasa_tr = datos3.iloc[orden[:210]]["abandona"].mean()
assert np.isclose(ll_base, log_loss(datos3.iloc[orden[210:]]["abandona"], np.full(90, tasa_tr))), "il riferimento usa il tasso di abbandono di train, non quello di test"
assert hay_sobreajuste == (ll_test > ll_train), "confronta ll_test con ll_train"
print(f"log-loss: train {ll_train:.3f}   test {ll_test:.3f}   riferimento costante {ll_base:.3f}   overfitting? {hay_sobreajuste}")
print("Esercizio 3 corretto ✔")

## Esercizio 4 — Validazione incrociata 5-fold a mano

Con 300 clienti (`datos4`) e l'ordine casuale `orden4`, costruisci: `folds` (una lista di 5 array di indici, con `np.array_split(orden4, 5)`), `perdidas` (lista con il log-loss di ogni blocco: il modello `formula4` si adatta con gli **altri quattro blocchi** e si valuta sul blocco di test; usa un ciclo `for i in range(5)` e `np.concatenate`), `media_cv` (media delle 5 perdite) e `rango_bloques` (massimo meno minimo delle 5 perdite).

In [ ]:
datos4 = generar_clientes(300, np.random.default_rng(5))
formula4 = "abandona ~ antiguedad + gasto + tickets + plan_anual"
orden4 = np.random.default_rng(1).permutation(300)

folds = None            # TODO: np.array_split(orden4, 5)
perdidas = None         # TODO: lista con il log-loss di ogni blocco (ciclo for)
media_cv = None         # TODO
rango_bloques = None    # TODO

# Controllo automatico
assert all(v is not None for v in (folds, perdidas, media_cv, rango_bloques)), "Completa le righe contrassegnate con TODO"
assert len(folds) == 5 and all(len(f) == 60 for f in folds), "5 blocchi da 60 clienti ciascuno"
assert sorted(np.concatenate(folds)) == list(range(300)), "ogni cliente deve stare in un solo blocco"
ref_p = []
for i in range(5):
    te_r = datos4.iloc[orden4[i * 60:(i + 1) * 60]]
    tr_r = datos4.iloc[np.concatenate([orden4[j * 60:(j + 1) * 60] for j in range(5) if j != i])]
    ref_p.append(log_loss(te_r["abandona"], smf.logit(formula4, data=tr_r).fit(disp=0).predict(te_r)))
assert len(perdidas) == 5 and np.allclose(perdidas, ref_p), "ogni perdita: adatta con gli altri 4 blocchi e valuta sul blocco di test"
assert np.isclose(media_cv, np.mean(ref_p)), "media_cv = media delle 5 perdite"
assert np.isclose(rango_bloques, max(ref_p) - min(ref_p)), "rango_bloques = massimo - minimo"
print("Log-loss per blocco:", np.round(perdidas, 3), " media:", round(media_cv, 3), " intervallo:", round(rango_bloques, 3))
print("Esercizio 4 corretto ✔")

## Esercizio 5 — Accuratezza contro la previsione banale e matrice di confusione

Un modello logistico (`m5`, adattato con 700 clienti) prevede le probabilità `p5` per 300 clienti di test, di cui solo il 12 % abbandona (`y_test5`). Classifica "abbandona" se `p5 > 0.5` e calcola, **a mano** (con `numpy`): `vp`, `fp`, `fn`, `vn` (interi), `exactitud5`, `precision5`, `recall5`, `exactitud_base5` (quella della previsione "nessuno abbandona") e `supera_base` (`True` se `exactitud5 > exactitud_base5`). Poi ripeti con la soglia 0.2: `exactitud_02` e `supera_base_02`.

In [ ]:
rng_e5 = np.random.default_rng(55)
n5 = 1000
x1, x2 = rng_e5.normal(size=n5), rng_e5.normal(size=n5)
y5 = rng_e5.binomial(1, 1 / (1 + np.exp(-(-2.8 + 1.6 * x1 + 0.8 * x2))))
X5 = sm.add_constant(np.column_stack([x1, x2]))
m5 = sm.Logit(y5[:700], X5[:700]).fit(disp=0)
p5, y_test5 = m5.predict(X5[700:]), y5[700:]

vp = None                # TODO: prevede 1 ed è 1
fp = None                # TODO
fn = None                # TODO
vn = None                # TODO
exactitud5 = None        # TODO
precision5 = None        # TODO
recall5 = None           # TODO
exactitud_base5 = None   # TODO
supera_base = None       # TODO
exactitud_02 = None      # TODO: accuratezza con soglia 0.2
supera_base_02 = None    # TODO

# Controllo automatico
assert all(v is not None for v in (vp, fp, fn, vn, exactitud5, precision5, recall5, exactitud_base5, supera_base, exactitud_02, supera_base_02)), "Completa le righe contrassegnate con TODO"
pr_ref = np.asarray(p5) > 0.5
assert (vp, fp, fn, vn) == (int(((pr_ref == 1) & (y_test5 == 1)).sum()), int(((pr_ref == 1) & (y_test5 == 0)).sum()), int(((pr_ref == 0) & (y_test5 == 1)).sum()), int(((pr_ref == 0) & (y_test5 == 0)).sum())), "VP: prevede 1 ed è 1; FP: prevede 1 ed è 0; FN: prevede 0 ed è 1; VN: prevede 0 ed è 0"
assert vp + fp + fn + vn == 300, "i quattro numeri sommano a 300"
assert np.isclose(exactitud5, (vp + vn) / 300), "accuratezza = (VP + VN) / totale"
assert np.isclose(precision5, vp / (vp + fp)), "precisione = VP / (VP + FP)"
assert np.isclose(recall5, vp / (vp + fn)), "recall = VP / (VP + FN)"
assert np.isclose(exactitud_base5, np.mean(y_test5 == 0)), "'nessuno abbandona' azzecca tutti quelli che non abbandonano"
assert supera_base == (exactitud5 > exactitud_base5), "confronta exactitud5 con exactitud_base5"
assert np.isclose(exactitud_02, np.mean((np.asarray(p5) > 0.2) == y_test5)), "ripeti l'accuratezza con p5 > 0.2"
assert supera_base_02 == (exactitud_02 > exactitud_base5) and supera_base_02 == False, "con la soglia 0.2 l'accuratezza resta sotto la previsione banale, anche se il recall sale"
print(f"VP={vp} FP={fp} FN={fn} VN={vn}   accuratezza {exactitud5:.3f} (base {exactitud_base5:.3f})   precisione {precision5:.3f}   recall {recall5:.3f}")
print("Esercizio 5 corretto ✔")

---
## Sfida finale — Scegliere un modello con criterio e validarlo

Con 3 000 clienti nuovi, separa il 70 % per l'addestramento e il 30 % per il test. Adatta con `train_r` tre modelli candidati (`formulas`) e calcola: `aic_por_modelo` (AIC di ciascuno, su `train_r`), `ll_test_por_modelo` (log-loss di ciascuno su `test_r`), `modelo_elegido` (il nome di quello con AIC minore) e `mejora_vs_base` (quanto migliora il log-loss in test di quello scelto rispetto alla previsione costante, `ll_base − ll_elegido`, positivo se migliora). Scrivi in `conclusion` una frase che spieghi la tua scelta (minimo 40 caratteri).

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
orden_r = rng_r.permutation(3000)
train_r, test_r = datos_r.iloc[orden_r[:2100]], datos_r.iloc[orden_r[2100:]]
formulas = {
    "A": "abandona ~ antiguedad + tickets",
    "B": "abandona ~ antiguedad + tickets + plan_anual + gasto",
    "C": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}

aic_por_modelo = None      # TODO: dict {nome: AIC} adattando ogni formula con train_r
ll_test_por_modelo = None  # TODO: dict {nome: log-loss su test}
modelo_elegido = None      # TODO
mejora_vs_base = None      # TODO
conclusion = None          # TODO

# Controllo automatico
assert all(v is not None for v in (aic_por_modelo, ll_test_por_modelo, modelo_elegido, mejora_vs_base, conclusion)), "Completa le righe contrassegnate con TODO"
ref_m = {k: smf.logit(f, data=train_r).fit(disp=0) for k, f in formulas.items()}
assert all(np.isclose(aic_por_modelo[k], ref_m[k].aic) for k in formulas), "l'AIC si calcola su train con .aic"
assert all(np.isclose(ll_test_por_modelo[k], log_loss(test_r["abandona"], ref_m[k].predict(test_r))) for k in formulas), "valuta su test i modelli adattati con train"
assert modelo_elegido == min(ref_m, key=lambda k: ref_m[k].aic), "scegli il nome con AIC minore"
ll_b = log_loss(test_r["abandona"], np.full(len(test_r), train_r["abandona"].mean()))
assert np.isclose(mejora_vs_base, ll_b - ll_test_por_modelo[modelo_elegido]), "miglioramento = log-loss costante − log-loss del modello scelto, entrambi su test"
assert mejora_vs_base > 0, "il modello scelto dovrebbe migliorare la previsione costante"
assert isinstance(conclusion, str) and len(conclusion) >= 40, "scrivi una frase di almeno 40 caratteri"
print("AIC:", {k: round(v, 1) for k, v in aic_por_modelo.items()})
print("log-loss test:", {k: round(v, 4) for k, v in ll_test_por_modelo.items()}, "scelto:", modelo_elegido)
print("Sfida finale corretta ✔")

---
## Chiusura della pratica

Hai percorso il ciclo di scelta e validazione di un modello: rilevare la multicollinearità, confrontare candidati con AIC e BIC, valutare con hold-out e con k-fold, e leggere una matrice di confusione con classi diseguali. Prosegui con la lezione 13: Test A/B e disegno degli esperimenti.

---
## Soluzioni (non guardare prima di aver provato!)

Ogni cella seguente contiene la soluzione di un esercizio. Puoi eseguirle per confrontarle con il tuo risultato.

### Soluzione dell'esercizio 1

In [ ]:
datos = generar_clientes(1500, np.random.default_rng(2026))
columnas = ["antiguedad", "gasto", "datos_gb", "tickets", "plan_anual"]
X_e1 = sm.add_constant(datos[columnas])

vifs = {c: variance_inflation_factor(X_e1.values, i) for i, c in enumerate(X_e1.columns) if c != "const"}
var_a_quitar = "datos_gb"
se_con = smf.logit("abandona ~ antiguedad + gasto + datos_gb + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
se_sin = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]

assert all(v is not None for v in (vifs, var_a_quitar, se_con, se_sin)), "Completa le righe contrassegnate con TODO"
assert set(vifs) == set(columnas), "vifs deve avere una voce per variabile, senza la costante"
ref = {c: variance_inflation_factor(X_e1.values, i) for i, c in enumerate(X_e1.columns) if c != "const"}
assert all(np.isclose(vifs[c], ref[c]) for c in columnas), "usa variance_inflation_factor(X.values, i) sulla matrice con costante"
assert var_a_quitar == "datos_gb", "elimina datos_gb: gasto è la fatturazione, che l'azienda controlla"
ref_con = smf.logit("abandona ~ antiguedad + gasto + datos_gb + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
ref_sin = smf.logit("abandona ~ antiguedad + gasto + tickets + plan_anual", data=datos).fit(disp=0).bse["gasto"]
assert np.isclose(se_con, ref_con) and np.isclose(se_sin, ref_sin), "adatta i due modelli con smf.logit e prendi .bse['gasto']"
assert se_sin < se_con / 2, "eliminando la variabile ridondante l'errore standard dovrebbe scendere molto"
print({k: round(v, 1) for k, v in vifs.items()})
print(f"Errore standard di gasto: {se_con:.4f} → {se_sin:.4f}")
print("Esercizio 1 corretto ✔")

### Soluzione dell'esercizio 2

In [ ]:
datos2 = generar_clientes(500, np.random.default_rng(100))
formulas2 = {"M2": "abandona ~ antiguedad + tickets + plan_anual",
             "M3": "abandona ~ antiguedad + tickets + plan_anual + gasto",
             "M4": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}
ajustes2 = {nombre: smf.logit(f, data=datos2).fit(disp=0) for nombre, f in formulas2.items()}

aic_mano = {nombre: 2 * len(m.params) - 2 * m.llf for nombre, m in ajustes2.items()}
bic_mano = {nombre: len(m.params) * np.log(m.nobs) - 2 * m.llf for nombre, m in ajustes2.items()}
elegido_aic = min(aic_mano, key=aic_mano.get)
elegido_bic = min(bic_mano, key=bic_mano.get)
dif_irrelevante = bool(abs(aic_mano["M2"] - aic_mano["M3"]) < 2)

assert all(v is not None for v in (aic_mano, bic_mano, elegido_aic, elegido_bic, dif_irrelevante)), "Completa le righe contrassegnate con TODO"
assert all(np.isclose(aic_mano[k], ajustes2[k].aic) for k in formulas2), "AIC = 2k - 2LL, con k = len(m.params) (include la costante)"
assert all(np.isclose(bic_mano[k], ajustes2[k].bic) for k in formulas2), "BIC = k·ln(n) - 2LL, con n = m.nobs"
assert elegido_aic == min(ajustes2, key=lambda k: ajustes2[k].aic) and elegido_bic == min(ajustes2, key=lambda k: ajustes2[k].bic), "scegli il nome con il valore minore di ciascun criterio"
assert elegido_aic == "M3" and elegido_bic == "M2", "in questo campione l'AIC preferisce M3 e il BIC preferisce il più semplice, M2"
assert dif_irrelevante == True, "gli AIC di M2 e M3 differiscono di meno di 2 punti"
print("AIC:", {k: round(v, 1) for k, v in aic_mano.items()}, "→", elegido_aic)
print("BIC:", {k: round(v, 1) for k, v in bic_mano.items()}, "→", elegido_bic)
print("Esercizio 2 corretto ✔")

### Soluzione dell'esercizio 3

In [ ]:
rng_e3 = np.random.default_rng(77)
datos3 = generar_clientes(300, rng_e3)
for j in range(15):
    datos3[f"nz{j + 1}"] = rng_e3.normal(size=300)
formula3 = "abandona ~ antiguedad + gasto + tickets + plan_anual" + "".join(f" + nz{j + 1}" for j in range(15))
orden = rng_e3.permutation(300)

train = datos3.iloc[orden[:210]]
test = datos3.iloc[orden[210:]]
m_e3 = smf.logit(formula3, data=train).fit(disp=0, maxiter=200)
ll_train = log_loss(train["abandona"], m_e3.predict(train))
ll_test = log_loss(test["abandona"], m_e3.predict(test))
ll_base = log_loss(test["abandona"], np.full(len(test), train["abandona"].mean()))
hay_sobreajuste = bool(ll_test > ll_train)

assert all(v is not None for v in (train, test, ll_train, ll_test, ll_base, hay_sobreajuste)), "Completa le righe contrassegnate con TODO"
assert len(train) == 210 and len(test) == 90, "70 % per l'addestramento (210) e 30 % per il test (90)"
assert set(train.index).isdisjoint(test.index), "addestramento e test non devono condividere righe"
m3_ref = smf.logit(formula3, data=datos3.iloc[orden[:210]]).fit(disp=0, maxiter=200)
assert np.isclose(ll_train, log_loss(datos3.iloc[orden[:210]]["abandona"], m3_ref.predict(datos3.iloc[orden[:210]]))), "ll_train: log_loss(train['abandona'], modelo.predict(train))"
assert np.isclose(ll_test, log_loss(datos3.iloc[orden[210:]]["abandona"], m3_ref.predict(datos3.iloc[orden[210:]]))), "il modello si adatta solo con train e si valuta su test"
tasa_tr = datos3.iloc[orden[:210]]["abandona"].mean()
assert np.isclose(ll_base, log_loss(datos3.iloc[orden[210:]]["abandona"], np.full(90, tasa_tr))), "il riferimento usa il tasso di abbandono di train, non quello di test"
assert hay_sobreajuste == (ll_test > ll_train), "confronta ll_test con ll_train"
print(f"log-loss: train {ll_train:.3f}   test {ll_test:.3f}   riferimento costante {ll_base:.3f}   overfitting? {hay_sobreajuste}")
print("Esercizio 3 corretto ✔")

### Soluzione dell'esercizio 4

In [ ]:
datos4 = generar_clientes(300, np.random.default_rng(5))
formula4 = "abandona ~ antiguedad + gasto + tickets + plan_anual"
orden4 = np.random.default_rng(1).permutation(300)

folds = np.array_split(orden4, 5)
perdidas = []
for i in range(5):
    test_i = datos4.iloc[folds[i]]
    train_i = datos4.iloc[np.concatenate([folds[j] for j in range(5) if j != i])]
    m_i = smf.logit(formula4, data=train_i).fit(disp=0)
    perdidas.append(log_loss(test_i["abandona"], m_i.predict(test_i)))
media_cv = float(np.mean(perdidas))
rango_bloques = float(max(perdidas) - min(perdidas))

assert all(v is not None for v in (folds, perdidas, media_cv, rango_bloques)), "Completa le righe contrassegnate con TODO"
assert len(folds) == 5 and all(len(f) == 60 for f in folds), "5 blocchi da 60 clienti ciascuno"
assert sorted(np.concatenate(folds)) == list(range(300)), "ogni cliente deve stare in un solo blocco"
ref_p = []
for i in range(5):
    te_r = datos4.iloc[orden4[i * 60:(i + 1) * 60]]
    tr_r = datos4.iloc[np.concatenate([orden4[j * 60:(j + 1) * 60] for j in range(5) if j != i])]
    ref_p.append(log_loss(te_r["abandona"], smf.logit(formula4, data=tr_r).fit(disp=0).predict(te_r)))
assert len(perdidas) == 5 and np.allclose(perdidas, ref_p), "ogni perdita: adatta con gli altri 4 blocchi e valuta sul blocco di test"
assert np.isclose(media_cv, np.mean(ref_p)), "media_cv = media delle 5 perdite"
assert np.isclose(rango_bloques, max(ref_p) - min(ref_p)), "rango_bloques = massimo - minimo"
print("Log-loss per blocco:", np.round(perdidas, 3), " media:", round(media_cv, 3), " intervallo:", round(rango_bloques, 3))
print("Esercizio 4 corretto ✔")

### Soluzione dell'esercizio 5

In [ ]:
rng_e5 = np.random.default_rng(55)
n5 = 1000
x1, x2 = rng_e5.normal(size=n5), rng_e5.normal(size=n5)
y5 = rng_e5.binomial(1, 1 / (1 + np.exp(-(-2.8 + 1.6 * x1 + 0.8 * x2))))
X5 = sm.add_constant(np.column_stack([x1, x2]))
m5 = sm.Logit(y5[:700], X5[:700]).fit(disp=0)
p5, y_test5 = m5.predict(X5[700:]), y5[700:]

pred5 = np.asarray(p5) > 0.5
vp = int(((pred5 == 1) & (y_test5 == 1)).sum())
fp = int(((pred5 == 1) & (y_test5 == 0)).sum())
fn = int(((pred5 == 0) & (y_test5 == 1)).sum())
vn = int(((pred5 == 0) & (y_test5 == 0)).sum())
exactitud5 = (vp + vn) / len(y_test5)
precision5 = vp / (vp + fp)
recall5 = vp / (vp + fn)
exactitud_base5 = float(np.mean(y_test5 == 0))
supera_base = bool(exactitud5 > exactitud_base5)
exactitud_02 = float(np.mean((np.asarray(p5) > 0.2) == y_test5))
supera_base_02 = bool(exactitud_02 > exactitud_base5)

assert all(v is not None for v in (vp, fp, fn, vn, exactitud5, precision5, recall5, exactitud_base5, supera_base, exactitud_02, supera_base_02)), "Completa le righe contrassegnate con TODO"
pr_ref = np.asarray(p5) > 0.5
assert (vp, fp, fn, vn) == (int(((pr_ref == 1) & (y_test5 == 1)).sum()), int(((pr_ref == 1) & (y_test5 == 0)).sum()), int(((pr_ref == 0) & (y_test5 == 1)).sum()), int(((pr_ref == 0) & (y_test5 == 0)).sum())), "VP: prevede 1 ed è 1; FP: prevede 1 ed è 0; FN: prevede 0 ed è 1; VN: prevede 0 ed è 0"
assert vp + fp + fn + vn == 300, "i quattro numeri sommano a 300"
assert np.isclose(exactitud5, (vp + vn) / 300), "accuratezza = (VP + VN) / totale"
assert np.isclose(precision5, vp / (vp + fp)), "precisione = VP / (VP + FP)"
assert np.isclose(recall5, vp / (vp + fn)), "recall = VP / (VP + FN)"
assert np.isclose(exactitud_base5, np.mean(y_test5 == 0)), "'nessuno abbandona' azzecca tutti quelli che non abbandonano"
assert supera_base == (exactitud5 > exactitud_base5), "confronta exactitud5 con exactitud_base5"
assert np.isclose(exactitud_02, np.mean((np.asarray(p5) > 0.2) == y_test5)), "ripeti l'accuratezza con p5 > 0.2"
assert supera_base_02 == (exactitud_02 > exactitud_base5) and supera_base_02 == False, "con la soglia 0.2 l'accuratezza resta sotto la previsione banale, anche se il recall sale"
print(f"VP={vp} FP={fp} FN={fn} VN={vn}   accuratezza {exactitud5:.3f} (base {exactitud_base5:.3f})   precisione {precision5:.3f}   recall {recall5:.3f}")
print("Esercizio 5 corretto ✔")

### Soluzione della sfida finale

In [ ]:
rng_r = np.random.default_rng(99)
datos_r = generar_clientes(3000, rng_r)
orden_r = rng_r.permutation(3000)
train_r, test_r = datos_r.iloc[orden_r[:2100]], datos_r.iloc[orden_r[2100:]]
formulas = {
    "A": "abandona ~ antiguedad + tickets",
    "B": "abandona ~ antiguedad + tickets + plan_anual + gasto",
    "C": "abandona ~ antiguedad + tickets + plan_anual + gasto + datos_gb"}

ajustes = {k: smf.logit(f, data=train_r).fit(disp=0) for k, f in formulas.items()}
aic_por_modelo = {k: m.aic for k, m in ajustes.items()}
ll_test_por_modelo = {k: log_loss(test_r["abandona"], m.predict(test_r)) for k, m in ajustes.items()}
modelo_elegido = min(aic_por_modelo, key=aic_por_modelo.get)
ll_base_r = log_loss(test_r["abandona"], np.full(len(test_r), train_r["abandona"].mean()))
mejora_vs_base = ll_base_r - ll_test_por_modelo[modelo_elegido]
conclusion = "Scelgo il modello con AIC minore, che ha anche senso di business, ed evito la variabile ridondante datos_gb; lo valido sul test."

assert all(v is not None for v in (aic_por_modelo, ll_test_por_modelo, modelo_elegido, mejora_vs_base, conclusion)), "Completa le righe contrassegnate con TODO"
ref_m = {k: smf.logit(f, data=train_r).fit(disp=0) for k, f in formulas.items()}
assert all(np.isclose(aic_por_modelo[k], ref_m[k].aic) for k in formulas), "l'AIC si calcola su train con .aic"
assert all(np.isclose(ll_test_por_modelo[k], log_loss(test_r["abandona"], ref_m[k].predict(test_r))) for k in formulas), "valuta su test i modelli adattati con train"
assert modelo_elegido == min(ref_m, key=lambda k: ref_m[k].aic), "scegli il nome con AIC minore"
ll_b = log_loss(test_r["abandona"], np.full(len(test_r), train_r["abandona"].mean()))
assert np.isclose(mejora_vs_base, ll_b - ll_test_por_modelo[modelo_elegido]), "miglioramento = log-loss costante − log-loss del modello scelto, entrambi su test"
assert mejora_vs_base > 0, "il modello scelto dovrebbe migliorare la previsione costante"
assert isinstance(conclusion, str) and len(conclusion) >= 40, "scrivi una frase di almeno 40 caratteri"
print("AIC:", {k: round(v, 1) for k, v in aic_por_modelo.items()})
print("log-loss test:", {k: round(v, 4) for k, v in ll_test_por_modelo.items()}, "scelto:", modelo_elegido)
print("Sfida finale corretta ✔")

---
## Autovalutazione

Annota in una cella di testo o su carta:

- **Quali esercizi ho risolto senza aiuto?**
- **Quale concetto resta poco chiaro?** (ad esempio: VIF, AIC contro BIC, k-fold, precisione e recall, data leakage)
- **Tempo dedicato:** ___ minuti
- **Una domanda che voglio portare alla prossima lezione:** ...